# The Orbit Determination Problem

## Introduction
By **orbit determination** we mean the process, or a set of techniques, for obtaining knowledge about the motion of objects such as moons, planets, and spacecraft relative to the center of mass of the Earth in a specified coordinate system.

Orbit determination for celestial bodies has been a general concern of astronomers and mathematicians since the beginning of civilization and indeed has attracted some of the best analytical minds to develop the basis for much of the fundamental mathematics in use today; orbit determination methods have been used throughout history and developed independently by many astronomers. This challenging methodology actually started on January 1 1801, when the Italian astronomer Giuseppe Piazzi discovered a planetoid (which he called *Ceres*) working from an observatory in Palermo, Italy. He was only able to observe the planetoid for 41 days, during which he collected 22 measurements forming an arc of only 9 degrees. Ceres was then lost, and the astronomers could no longer find it. There was now a challenge of calculating Ceres' orbit using only the observations Piazzi made, so that astronomers would be able to sight Ceres when it reemerged. 

The German mathematician Calr Friedrich Gauss (30 April 1777 – 23 February 1855) was the first able to solve this problem by developing its own process (called Gauss method) which made use of at least 3 observation to estimate future positions of Ceres (which was successfully recovered in late 1801. The Gauss method is still used as **preliminary orbit determination**, where there are no previous information on the state (position, velocity) of the body of interest.

$\quad$

<center><img src="Images/piazziGauss.png" alt="Drawing" style="width: 550px;"/><center>
    




To get the full story of this milestone in astronomy history, please take a look at the following web pages:

https://sites.math.rutgers.edu/~cherlin/History/Papers1999/weiss.html

https://archive.schillerinstitute.com/fid_97-01/982_orbit_ceres.pdf


## The State Estimation Process


The **state** of a dynamic system is the the current configuration or values of that system that is of interest to us. In other words, the state should be considered as the actual value of the system, but this value is usually hidden to us. For the orbit determination problem the minimal set of parameters shaping the system's state are the satellite's position and velocity vectors at some given epoch $t$:

$$ \mathbf{x}(t) = \begin{bmatrix}
\mathbf{r}(t)  \\
\mathbf{v}(t)
\end{bmatrix} $$

Orbit determination is an [inverse](../Observation_and_Estimation/01-The_Estimation_Problem.ipynb) problem: it is fundamentally concerned with **estimating the state of a spacecraft from a finite set of measurements**. The word 'estimate' means that the derivation of the state vector along time is subject to the uncertainties related to the models used in the process.

$\quad$

<center><img src="Images/od.png" alt="Drawing" style="width: 850px;"/><center>

The Figure above shows how the state estimation process takes place. The available measurements are the primary source of information to derive a state estimate from the observables through the application of two different models: 

1) the **force model**, describing the the forces acting on the body under investigation as well as the uncertanties related to the system's unknows, and 
2) the **sensor model**, which is composed by two conceptual elements: the **bias model**, providing a statistical representation of the noise affecting the tracking device and the **measurement model**, defining how the state is related to the observables


### Force Model

The spacecraft motion is described by the following non-linear differential equation:

$$\dot{\mathbf{x}} = f(\mathbf{x},t)$$

While numerical integration of the equations of motion provides a prediction of the 'nominal' spacecraft trajectory (**modeled state**), state propagation alone is insufficient for the estimation process, which aims to find the 'best possible' orbit for the object under investigation. What the deterministic equations of motion are missing is the complete, exact definition of the force model respect to the true set of forces surounding the satellite. 

Whichever model is used to describe the forces, there will always be two main source of errors:

1) The incomplete/inperfect definition of each constituent of the true forces (e.g., we model the Earth gravitational field with a finite number of harmonics, introducing a cumulative error along the propagation). The deviations between modeled and true spacecraft dynamics are considered by introducing **process noise** in the system

2) The ignorance about the actual value of of internal parameters (e.g., ballistic coefficient or clock drift); this is described by a **stochastic process**


So, we can think the force model as the sum of the 'nominal' model (the one used in the differential equations to propagate the state along time) and some 'perturbations' which affect the force model itself: 

$$\dot{\mathbf{x}} = f(\mathbf{x},t) + \bold w (t)$$

#### Perturbation Analysis

Orbit determination algorithms require knowledge of the sensitivity of the trajectory to perturbations in the estimated parameters. This sensitivity information is provided by the **variational equations**, obtained through the linearization of the nonlinear dynamics about a reference trajectory.

<center><img src="Images/perturbation.png" alt="Drawing" style="width: 550px;"/><center>



Linearization leads to the **system matrix** $\mathbf{A}$:

$$\mathbf{A}(t)=\frac{\partial f}{\partial \mathbf{x}}$$

, which describes the local sensitivity of the dynamics to perturbations in the state vector. 

The integration of the variational equations produces the **State Transition Matrix** $\bold \Phi(t,t_0)$ ,which establishes the relationship

$$\delta \mathbf{x}(t) = \bold \Phi(t,t_0)\,\delta \mathbf{x}(t_0)$$

The STM therefore represents the first-order mapping between perturbations at the initial epoch and perturbations at any subsequent time. Equivalently, it may be interpreted as the Jacobian of the propagated state with respect to the initial state,

$$\bold \Phi(t,t_0)= \frac{\partial \mathbf{x}(t)} {\partial \mathbf{x}(t_0)}$$

$\bold \Phi$ and $\bold A$ are related each other by 

$$\dot{\mathbf{x}} = \mathbf{f}(t,\mathbf{x}), \dot{\boldsymbol{\Phi}} = \mathbf{A}(t)\boldsymbol{\Phi}$$

and initial conditions

$$\mathbf{x}(t_0)=\mathbf{x}_0, \qquad \boldsymbol{\Phi}(t_0,t_0)=\mathbf{I}$$

The importance of the STM becomes evident considering that the covariance propagation relies on the relation

$$\bold P(t)=\bold \Phi(t,t_0) \bold P_0\bold \Phi^T(t,t_0)$$

, which maps the uncertainty associated with the estimated state through the dynamical system. Without the STM, it would be impossible to efficiently compute the corrections required by estimation algorithms or to rigorously characterize the growth and evolution of orbit uncertainty.



### Sensor Models

The quality of the estimated state  provided by the measurement depends largely on the quality of the models used in the process and the information contained into the measurements.

We here define an **observable** as the physical quantity that a sensor can measure and that can be mathematically related to the spacecraft state through the following relation:

$$\mathbf{y}=h(\mathbf{x},t)+\mathbf{\epsilon}$$

, where

- $\mathbf{x}$ = spacecraft state
- $h(\cdot)$ = **measurement model**
- $\mathbf{\epsilon}$ = measurement noise (described by a **bias model**)
- $\mathbf{y}$ = measurement (observable)

#### The Measurement Model
The measurement model describes how the observables are related to the state. For example, for a range-based measurement, the measurement itself is the numerical value reported by the sensor (e.g., $\rho = 3481.45$ $km$) and its measurement model is $\rho =  \left\| r_{sat}-r_{station}\right\|$.

The information provided by a sensor is mathematically quantified through the **sensitivity** of the observable respect to the state. The sensitivity matrix ((also called state to measurement matrix or measurement Jacobian) is

$$\mathbf H = \frac{\partial h}{\partial \mathbf x}$$

Each element indicates how strongly the measurement responds to changes in a particular state variable.
Large derivatives mean that the observable contains significant information regarding the corresponding state component, while small derivatives imply limited observability of that component.

#### The Bias Model

Sensors produce noisy measurement, so the measurement model has to be bundled with en error model. The error $\epsilon$ contained in every simgle measurement is unknown, and the only way to model the sensor's noise is to characterize it in statistical terms. The bias model for a sensor is composed by two terms:

1) A **stochastic process**, which describes the evolution over time of a system whose state is subject to random variations. Examples of stochastic process models are:

  - **Gauss-Markov**, used when the mean is known and the process is bounded. It represents an exponential decay from the estimate back to the known mean in absence of measurements. Examples: atmospheric density, tropospheric delay
  - **Vasicek**, used when the mean is unknown but the uncertainty around it is known. It represents an exponential decay from the estimate back to the estimated mean in absence of measurements. Examples: drag coefficient
  - **Random walk**: no mean value expected, not bounded process. Example: tracking station clock bias
2) A  **white noise** term. This is a zero-mean, uncorrelated random measurement error with constant power spectral density across all frequencies


## Classical Orbit Determination
The classical orbit determnation problem is characterized by the assumption that the involved bodies move under the influence of a central force (Keplerian orbits), even this assumption is away from the reality since the satellite motion is affected from disturbing forces (both conservative and non-conservative) that significantly alter the satellite’s status of the motion.

The fundamental equation of motion to consider in this case is:


**$$\mathbf{\ddot{r}}=-\frac{\mu}{r^3} \mathbf{r}$$ 

, where no disturbance forces are taken into account. The early methods to determine celestial body's trajectories where developed by Laplace (1780) and Gauss (1801). They both rely on aa set of three angular measurements from an observing site at three different times.

## Optimal Orbit Determination

The optimal orbit determination can be considered as the evolution of its classical counterpart. In this case the state estimate is *complete*: all the unknowns are part of the state estimate vector. This concept is quite important: consider for example a cooperative range tracking, where the signal sent from the radar is retransmitted by an onboard transponder. If the transponder delay is not taken into account, the range measurement is biased due to the longest interval between the transmission and the receiving time. The erroneous range estimate is then aliased in the other's orbit elements estimate, significantly degrading them.
There are other factors characterizing the optimal orbit determination, such as the usage of appropriate physics of sensor and force modeling (so, the two body model is not enough anymore) and sequential processing techniques.



## Orbit Determination Methods

Orbit determination methods can be classified in three categories according to their inputs, outputs, and accuracy performances:

### Initial orbit determination (IOD)

Methods characterized by the use of approximations  and/or recursive algorithms operating on nonlinear two-body dynamics.

Inputs:

- tracking measurements
- tracking platform locations

Outputs:

- spacecraft position and velocity estimates

### Batch least squares differential corrections (LS)

Method characterized by a sequence of linear LS corrections. At each correction, the LS algorithm processes a subset of a fixed batch of tracking measurements simultaneously.

Inputs:

- tracking measurements
- tracking platform locations
- a priori initial state estimate

Outputs:

- refined position and velocity estimates
- state error covariance estimate

### Sequential processing (SP)

Method characterized by a forward-time recursive process consisting of a repeating pattern of time update of the state and measurement estimates. 

Inputs:

- tracking measurements
- tracking platform locations
- a priori initial state estimate
- state error covariance estimate

Outputs:

- refined position and velocity estimates
- refined state error covariance estimate
